# Product Image Enhancer - Colab Notebook

This notebook is a standalone version of the image enhancement component. It lets you upload an inventory image, choose `studio_product`, `studio_product_generative`, `Real-ESRGAN x4plus`, or `pillow_fallback`, enhance the image, compare before/after, and export an API-style base64 response.

`studio_product_generative` uses OpenAI image editing, so Cell 5 will ask for `OPENAI_API_KEY` the first time you select that engine.

**Recommended Colab runtime:** `Runtime > Change runtime type > T4 GPU`.

CPU works, but Real-ESRGAN can be slow on larger images.

In [ ]:
# @title 1. Install dependencies
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SETUP_MARKER = Path("/content/.image_enhancer_setup_complete")

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

if SETUP_MARKER.exists():
    missing_packages = []
    if importlib.util.find_spec("cv2") is None:
        missing_packages.append("opencv-python-headless==4.10.0.84")
    if importlib.util.find_spec("onnxruntime") is None:
        missing_packages.append("onnxruntime==1.19.2")
    if importlib.util.find_spec("rembg") is None:
        missing_packages.append("rembg==2.0.57")
    if importlib.util.find_spec("openai") is None:
        missing_packages.append("openai>=2.0.0")
    if missing_packages:
        pip_install(*missing_packages)

    import numpy
    import torch
    import torchvision
    print("Dependencies already installed.")
    print("NumPy:", numpy.__version__)
    print("Torch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
else:
    # Colab often starts with a newer NumPy. Replacing it inside a live runtime can leave compiled
    # packages in a mixed ABI state, so this cell restarts the runtime once after installation.
    pip_install("--force-reinstall", "numpy==1.26.4")

    if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
        pip_install("torch", "torchvision")

    # BasicSR/Real-ESRGAN need a compact inference stack. We install their heavy training/face extras separately only if needed.
    pip_install(
        "pillow",
        "opencv-python-headless==4.10.0.84",
        "onnxruntime==1.19.2",
        "rembg==2.0.57",
        "openai>=2.0.0",
        "scipy==1.13.1",
        "scikit-image==0.24.0",
        "lmdb==2.2.0",
        "addict==2.4.0",
        "future==1.0.0",
        "requests==2.32.5",
        "tb-nightly==2.21.0a20251023",
        "tqdm==4.67.3",
        "yapf==0.43.0",
        "wheel==0.46.3",
        "Cython==3.2.4",
    )
    pip_install("--no-deps", "basicsr==1.4.2", "realesrgan==0.3.0")
    SETUP_MARKER.write_text("installed")

    print("Dependencies installed. Restarting the Colab runtime once to clear NumPy binary state.")
    print("After the runtime reconnects, run this cell again, then continue to Cell 2.")
    os.kill(os.getpid(), 9)


In [ ]:
# @title 2. Download Real-ESRGAN weights
from pathlib import Path
import urllib.request

WEIGHTS_PATH = Path("/content/RealESRGAN_x4plus.pth")
WEIGHTS_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

if not WEIGHTS_PATH.exists():
    print("Downloading RealESRGAN_x4plus weights. This is about 64 MB.")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print(f"Weights ready: {WEIGHTS_PATH} ({WEIGHTS_PATH.stat().st_size / (1024 * 1024):.1f} MB)")

In [ ]:
# @title 3. Define the enhancement pipeline
from __future__ import annotations

import base64
import importlib.util
import io
import os
import sys
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional

import numpy as np
import torch
import torchvision.transforms.functional as TVF
from PIL import Image, ImageChops, ImageDraw, ImageEnhance, ImageFilter, ImageOps, ImageStat, UnidentifiedImageError

# BasicSR 1.4.2 imports torchvision.transforms.functional_tensor, which was removed in newer torchvision.
# This shim keeps the notebook working on current Colab runtimes.
if "torchvision.transforms.functional_tensor" not in sys.modules:
    functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
    functional_tensor.rgb_to_grayscale = TVF.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

PresetName = Literal["product_standard", "product_detail", "product_soft"]
EngineName = Literal["realesrgan", "pillow_fallback", "studio_product", "studio_product_generative"]
REMBG_MODEL_DIR = Path("/content/weights/rembg")
REMBG_MODEL_NAME = "isnet-general-use"
OPENAI_IMAGE_MODEL = os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-2")
OPENAI_IMAGE_SIZE = os.getenv("OPENAI_IMAGE_SIZE", "1024x1024")
OPENAI_IMAGE_QUALITY = os.getenv("OPENAI_IMAGE_QUALITY", "high")

@dataclass(frozen=True)
class EnhancementResult:
    image_bytes: bytes
    width: int
    height: int
    engine: EngineName
    engine_label: str
    preset: PresetName

class ProductImageEnhancer:
    def __init__(self, weights_path: Path, max_input_pixels: int = 24_000_000, target_long_edge: int = 1800):
        self.weights_path = weights_path
        self.max_input_pixels = max_input_pixels
        self.target_long_edge = target_long_edge
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._realesrgan = None
        self._rembg_session = None
        self._rembg_session_model: Optional[str] = None

    def enhance(self, file_bytes: bytes, preset: PresetName = "product_standard", engine: EngineName = "pillow_fallback") -> EnhancementResult:
        image = self._decode(file_bytes)
        if engine == "realesrgan":
            enhanced = self._enhance_with_realesrgan(image, preset)
            engine_label = "RealESRGAN_x4plus"
        elif engine == "studio_product":
            enhanced = self._studio_product_enhance(image, preset)
            engine_label = "Studio product"
        elif engine == "studio_product_generative":
            enhanced = self._studio_product_generative_enhance(image, preset)
            engine_label = f"Studio product generative ({OPENAI_IMAGE_MODEL})"
        else:
            enhanced = self._fallback_enhance(image, preset)
            engine_label = "Pillow fallback"

        output = self._encode(enhanced)
        return EnhancementResult(output, enhanced.width, enhanced.height, engine, engine_label, preset)

    def _decode(self, file_bytes: bytes) -> Image.Image:
        if not file_bytes:
            raise ValueError("Upload an image before enhancing.")
        try:
            image = Image.open(io.BytesIO(file_bytes))
            image = ImageOps.exif_transpose(image)
            image.load()
        except (UnidentifiedImageError, OSError) as exc:
            raise ValueError("The uploaded file is not a readable image.") from exc
        if image.width * image.height > self.max_input_pixels:
            raise ValueError("Image is too large. Use a file under 24 megapixels.")
        if image.mode not in ("RGB", "RGBA"):
            image = image.convert("RGB")
        if image.mode == "RGBA":
            background = Image.new("RGB", image.size, (255, 255, 255))
            background.paste(image, mask=image.getchannel("A"))
            image = background
        return image.convert("RGB")

    def _load_realesrgan(self):
        if self._realesrgan is not None:
            return self._realesrgan
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        self._realesrgan = RealESRGANer(
            scale=4,
            model_path=str(self.weights_path),
            model=model,
            tile=256 if self.device.type == "cuda" else 128,
            tile_pad=10,
            pre_pad=0,
            half=self.device.type == "cuda",
            device=self.device,
        )
        return self._realesrgan

    def _enhance_with_realesrgan(self, image: Image.Image, preset: PresetName) -> Image.Image:
        upsampler = self._load_realesrgan()
        rgb = np.array(image)
        bgr = rgb[:, :, ::-1]
        output_bgr, _ = upsampler.enhance(bgr, outscale=2)
        output_rgb = output_bgr[:, :, ::-1]
        enhanced = self._fit_long_edge(Image.fromarray(output_rgb), self.target_long_edge)
        return self._final_polish(enhanced, preset, denoise=False, strength="model")

    def _fallback_enhance(self, image: Image.Image, preset: PresetName) -> Image.Image:
        image = self._fit_long_edge(image, self.target_long_edge)
        return self._final_polish(image, preset, denoise=True, strength="fallback")

    def _studio_product_enhance(self, image: Image.Image, preset: PresetName) -> Image.Image:
        mask = self._segment_product_mask(image)
        if mask is None:
            return self._fallback_enhance(image, preset)

        bbox = mask.getbbox()
        if bbox is None:
            return self._fallback_enhance(image, preset)
        bbox = self._expand_box(bbox, image.size, padding=max(4, round(max(image.size) * 0.015)))
        product = image.crop(bbox)
        product_mask = mask.crop(bbox)
        product = self._final_polish(product, preset, denoise=True, strength="studio")
        product_rgba = product.convert("RGBA")
        product_rgba.putalpha(product_mask)
        canvas_size = self.target_long_edge
        product_rgba = self._fit_product_on_canvas(product_rgba, canvas_size)
        product_alpha = product_rgba.getchannel("A")
        canvas = self._studio_background(canvas_size)
        product_x = (canvas_size - product_rgba.width) // 2
        product_y = self._studio_product_y(canvas_size, product_rgba.height)
        shadow = self._build_shadow(product_alpha, canvas_size)
        shadow_x = (canvas_size - shadow.width) // 2
        shadow_y = min(canvas_size - shadow.height - max(8, canvas_size // 90), product_y + product_rgba.height - shadow.height // 2)
        canvas.paste(shadow, (shadow_x, shadow_y), shadow)
        canvas.paste(product_rgba, (product_x, product_y), product_alpha)
        return canvas.convert("RGB")

    def _studio_product_generative_enhance(self, image: Image.Image, preset: PresetName) -> Image.Image:
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError("Set OPENAI_API_KEY to use studio_product_generative.")
        if importlib.util.find_spec("openai") is None:
            raise ValueError("Install the openai Python package to use studio_product_generative.")

        reference = self._studio_product_enhance(image, preset)
        generated = self._enhance_with_openai_image_edit(reference)
        generated = generated.convert("RGB")
        generated = self._fit_long_edge(generated, self.target_long_edge)
        return self._final_polish(generated, preset, denoise=False, strength="model")

    def _enhance_with_openai_image_edit(self, reference: Image.Image) -> Image.Image:
        from openai import OpenAI

        client = OpenAI()
        reference_file = io.BytesIO(self._image_to_png_bytes(reference))
        reference_file.name = "product-reference.png"
        result = client.images.edit(
            model=OPENAI_IMAGE_MODEL,
            image=reference_file,
            prompt=self._generative_product_prompt(),
            quality=OPENAI_IMAGE_QUALITY,
            response_format="b64_json",
            size=OPENAI_IMAGE_SIZE,
        )
        if not result.data or not result.data[0].b64_json:
            raise RuntimeError("OpenAI image edit did not return image bytes.")
        output_bytes = base64.b64decode(result.data[0].b64_json)
        output = Image.open(io.BytesIO(output_bytes))
        output.load()
        return output

    def _generative_product_prompt(self) -> str:
        return (
            "Create a premium ecommerce product photo from this reference image. "
            "Preserve the exact product identity, shape, proportions, colors, material texture, logos, labels, "
            "printed text, packaging edges, and item count. Do not invent, replace, rewrite, remove, or stylize "
            "any branding or product text. Improve only the catalog photography: clean off-white studio background, "
            "balanced DSLR or mirrorless camera lighting, natural soft contact shadow, crisp focus, realistic color, "
            "and subtle high-end product-photo finish. Keep the product centered and fully visible. "
            "No hands, props, extra objects, decorative elements, watermarks, fake labels, or lifestyle scene."
        )

    def _image_to_png_bytes(self, image: Image.Image) -> bytes:
        buffer = io.BytesIO()
        image.save(buffer, format="PNG", optimize=True)
        buffer.seek(0)
        return buffer.getvalue()

    def _segment_product_mask(self, image: Image.Image) -> Optional[Image.Image]:
        mask = self._build_rembg_product_mask(image)
        if mask is not None:
            return mask

        mask = self._build_grabcut_product_mask(image)
        if mask is not None:
            return mask

        return self._build_heuristic_product_mask(image)

    def _build_rembg_product_mask(self, image: Image.Image) -> Optional[Image.Image]:
        try:
            from rembg import new_session, remove
        except Exception:
            return None

        try:
            REMBG_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            os.environ.setdefault("U2NET_HOME", str(REMBG_MODEL_DIR.resolve()))
            if self._rembg_session is None or self._rembg_session_model != REMBG_MODEL_NAME:
                self._rembg_session = new_session(REMBG_MODEL_NAME, providers=["CPUExecutionProvider"])
                self._rembg_session_model = REMBG_MODEL_NAME
            output = remove(image, session=self._rembg_session, post_process_mask=True)
            if output.mode != "RGBA":
                output = output.convert("RGBA")
            mask = self._clean_product_mask(output.getchannel("A"), image.size)
            return self._validate_product_mask(mask, image.size, allow_border_touch=True)
        except Exception as exc:
            print(f"rembg segmentation failed; using fallback: {exc}")
            return None

    def _build_grabcut_product_mask(self, image: Image.Image) -> Optional[Image.Image]:
        try:
            import cv2
        except Exception:
            return None

        try:
            max_side = 900
            scale = min(1.0, max_side / max(image.size))
            work = image
            if scale < 1.0:
                work = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.LANCZOS)
            rgb = np.array(work)
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            height, width = bgr.shape[:2]
            gc_mask = np.full((height, width), cv2.GC_PR_BGD, dtype=np.uint8)
            border_x = max(3, round(width * 0.035))
            border_y = max(3, round(height * 0.035))
            gc_mask[:border_y, :] = cv2.GC_BGD
            gc_mask[height - border_y :, :] = cv2.GC_BGD
            gc_mask[:, :border_x] = cv2.GC_BGD
            gc_mask[:, width - border_x :] = cv2.GC_BGD
            heuristic = self._build_heuristic_product_mask(work)
            if heuristic is not None:
                heuristic_array = np.array(heuristic)
                gc_mask[heuristic_array > 48] = cv2.GC_PR_FGD
                gc_mask[heuristic_array > 220] = cv2.GC_FGD
                mode = cv2.GC_INIT_WITH_MASK
                rect = (0, 0, 1, 1)
            else:
                rect = (max(1, round(width * 0.05)), max(1, round(height * 0.05)), max(2, round(width * 0.9)), max(2, round(height * 0.9)))
                mode = cv2.GC_INIT_WITH_RECT
            bgd_model = np.zeros((1, 65), np.float64)
            fgd_model = np.zeros((1, 65), np.float64)
            cv2.grabCut(bgr, gc_mask, rect, bgd_model, fgd_model, 5, mode)
            foreground = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 255, 0).astype("uint8")
            kernel_size = max(3, round(min(width, height) * 0.01))
            if kernel_size % 2 == 0:
                kernel_size += 1
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
            foreground = cv2.morphologyEx(foreground, cv2.MORPH_CLOSE, kernel, iterations=2)
            foreground = cv2.morphologyEx(foreground, cv2.MORPH_OPEN, kernel, iterations=1)
            mask = Image.fromarray(foreground, mode="L")
            if mask.size != image.size:
                mask = mask.resize(image.size, Image.Resampling.LANCZOS)
            mask = self._clean_product_mask(mask, image.size)
            return self._validate_product_mask(mask, image.size, allow_border_touch=False)
        except Exception as exc:
            print(f"OpenCV GrabCut segmentation failed; using fallback: {exc}")
            return None

    def _build_heuristic_product_mask(self, image: Image.Image) -> Optional[Image.Image]:
        background = Image.new("RGB", image.size, self._estimate_background_color(image))
        diff = ImageChops.difference(image, background).convert("L")
        stats = ImageStat.Stat(diff)
        threshold = max(18, min(58, round(stats.mean[0] + stats.stddev[0] * 0.75)))
        mask = diff.point(lambda pixel: 255 if pixel > threshold else 0, mode="L")
        mask = mask.filter(ImageFilter.MedianFilter(size=5))
        mask = mask.filter(ImageFilter.MaxFilter(size=9))
        mask = mask.filter(ImageFilter.MinFilter(size=5))
        mask = mask.filter(ImageFilter.GaussianBlur(radius=1.1))
        return self._validate_product_mask(mask, image.size, allow_border_touch=False)

    def _clean_product_mask(self, mask: Image.Image, image_size: tuple[int, int]) -> Image.Image:
        try:
            import cv2
        except Exception:
            return mask.convert("L").filter(ImageFilter.GaussianBlur(radius=0.8))

        mask_array = np.array(mask.convert("L"))
        hard_mask = (mask_array > 32).astype("uint8")
        component_count, labels, stats, _ = cv2.connectedComponentsWithStats(hard_mask, 8)
        if component_count <= 1:
            return Image.fromarray((hard_mask * 255).astype("uint8"), mode="L")
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest_area = int(areas.max()) if len(areas) else 0
        min_area = max(round(image_size[0] * image_size[1] * 0.003), round(largest_area * 0.08), 24)
        cleaned = np.zeros_like(hard_mask)
        for label in range(1, component_count):
            left = stats[label, cv2.CC_STAT_LEFT]
            top = stats[label, cv2.CC_STAT_TOP]
            width = stats[label, cv2.CC_STAT_WIDTH]
            height = stats[label, cv2.CC_STAT_HEIGHT]
            area = stats[label, cv2.CC_STAT_AREA]
            right = left + width
            bottom = top + height
            touches_border = left <= 1 or top <= 1 or right >= image_size[0] - 1 or bottom >= image_size[1] - 1
            if area < min_area:
                continue
            if touches_border and area < largest_area * 0.65:
                continue
            cleaned[labels == label] = 255
        if not cleaned.any():
            cleaned[labels == int(areas.argmax()) + 1] = 255
        kernel_size = max(3, round(min(image_size) * 0.004))
        if kernel_size % 2 == 0:
            kernel_size += 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel, iterations=1)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel, iterations=1)
        return Image.fromarray(cleaned.astype("uint8"), mode="L").filter(ImageFilter.GaussianBlur(radius=0.9))

    def _validate_product_mask(self, mask: Image.Image, image_size: tuple[int, int], allow_border_touch: bool) -> Optional[Image.Image]:
        mask = mask.convert("L").filter(ImageFilter.GaussianBlur(radius=0.6))
        histogram = mask.histogram()
        total_pixels = image_size[0] * image_size[1]
        coverage = sum(count for value, count in enumerate(histogram) if value > 24) / total_pixels
        if coverage < 0.02 or coverage > 0.88:
            return None
        hard_mask = mask.point(lambda pixel: 255 if pixel > 24 else 0, mode="L")
        bbox = hard_mask.getbbox()
        if bbox is None:
            return None
        box_width = bbox[2] - bbox[0]
        box_height = bbox[3] - bbox[1]
        if box_width >= image_size[0] * 0.98 and box_height >= image_size[1] * 0.98:
            return None
        bbox_area_ratio = (box_width * box_height) / total_pixels
        if not allow_border_touch and box_width > image_size[0] * 0.75 and box_height > image_size[1] * 0.75:
            if coverage < 0.45 and bbox_area_ratio > 0.55:
                return None
        if not allow_border_touch:
            border = max(2, round(min(image_size) * 0.025))
            top_band = hard_mask.crop((0, 0, image_size[0], border))
            bottom_band = hard_mask.crop((0, image_size[1] - border, image_size[0], image_size[1]))
            left_band = hard_mask.crop((0, 0, border, image_size[1]))
            right_band = hard_mask.crop((image_size[0] - border, 0, image_size[0], image_size[1]))
            border_pixels = sum(1 for band in (top_band, bottom_band, left_band, right_band) for pixel in band.getdata() if pixel)
            border_area = top_band.width * top_band.height + bottom_band.width * bottom_band.height
            border_area += left_band.width * left_band.height + right_band.width * right_band.height
            border_coverage = border_pixels / max(1, border_area)
            edge_hits = sum([bbox[0] <= border, bbox[1] <= border, bbox[2] >= image_size[0] - border, bbox[3] >= image_size[1] - border])
            if edge_hits >= 2 and (coverage > 0.18 or border_coverage > 0.08):
                return None
        return mask

    def _estimate_background_color(self, image: Image.Image) -> tuple[int, int, int]:
        width, height = image.size
        strip = max(1, round(min(width, height) * 0.04))
        regions = [
            image.crop((0, 0, width, strip)),
            image.crop((0, height - strip, width, height)),
            image.crop((0, 0, strip, height)),
            image.crop((width - strip, 0, width, height)),
        ]
        colors = [region.resize((1, 1), Image.Resampling.BOX).getpixel((0, 0)) for region in regions]
        return tuple(round(sum(color[channel] for color in colors) / len(colors)) for channel in range(3))

    def _expand_box(self, box: tuple[int, int, int, int], image_size: tuple[int, int], padding: int) -> tuple[int, int, int, int]:
        left, top, right, bottom = box
        width, height = image_size
        return (max(0, left - padding), max(0, top - padding), min(width, right + padding), min(height, bottom + padding))

    def _fit_product_on_canvas(self, product: Image.Image, canvas_size: int) -> Image.Image:
        max_width = round(canvas_size * 0.78)
        max_height = round(canvas_size * 0.76)
        scale = min(max_width / product.width, max_height / product.height)
        size = (max(1, round(product.width * scale)), max(1, round(product.height * scale)))
        return product.resize(size, Image.Resampling.LANCZOS)

    def _studio_product_y(self, canvas_size: int, product_height: int) -> int:
        top_limit = round(canvas_size * 0.08)
        bottom_limit = canvas_size - product_height - round(canvas_size * 0.13)
        centered = round((canvas_size - product_height) * 0.46)
        return max(top_limit, min(centered, bottom_limit))

    def _studio_background(self, size: int) -> Image.Image:
        top = (250, 251, 249)
        bottom = (239, 242, 240)
        strip = Image.new("RGB", (1, size), top)
        pixels = strip.load()
        for y in range(size):
            ratio = y / max(1, size - 1)
            pixels[0, y] = tuple(round(top[channel] * (1 - ratio) + bottom[channel] * ratio) for channel in range(3))
        return strip.resize((size, size), Image.Resampling.BOX)

    def _build_shadow(self, alpha: Image.Image, canvas_size: int) -> Image.Image:
        shadow_width = max(1, round(alpha.width * 0.9))
        shadow_height = max(18, round(canvas_size * 0.055))
        shadow = Image.new("L", (shadow_width, shadow_height), 0)
        draw = ImageDraw.Draw(shadow)
        inset_x = max(1, round(shadow_width * 0.08))
        inset_y = max(1, round(shadow_height * 0.22))
        draw.ellipse((inset_x, inset_y, shadow_width - inset_x, shadow_height - inset_y), fill=255)
        shadow = shadow.filter(ImageFilter.GaussianBlur(radius=max(10, canvas_size // 70)))
        shadow = shadow.point(lambda pixel: round(pixel * 0.20))
        shadow_layer = Image.new("RGBA", shadow.size, (18, 22, 20, 0))
        shadow_layer.putalpha(shadow)
        return shadow_layer

    def _final_polish(self, image: Image.Image, preset: PresetName, denoise: bool, strength: str) -> Image.Image:
        image = ImageOps.autocontrast(image, cutoff=1)
        if preset == "product_detail":
            color, contrast, sharpness = 1.04, 1.08, 1.26
            blur_radius, blend = 0.7, 0.28
        elif preset == "product_soft":
            color, contrast, sharpness = 1.01, 1.03, 1.08
            blur_radius, blend = 0.5, 0.18
        else:
            color, contrast, sharpness = 1.02, 1.05, 1.16
            blur_radius, blend = 0.6, 0.22
        if strength == "model":
            sharpness = max(1.02, sharpness - 0.06)
            blend *= 0.5
        if denoise:
            smoothed = image.filter(ImageFilter.MedianFilter(size=3))
            image = Image.blend(image, smoothed, blend)
        image = image.filter(ImageFilter.UnsharpMask(radius=blur_radius, percent=120, threshold=3))
        image = ImageEnhance.Contrast(image).enhance(contrast)
        image = ImageEnhance.Color(image).enhance(color)
        image = ImageEnhance.Sharpness(image).enhance(sharpness)
        return image

    def _fit_long_edge(self, image: Image.Image, long_edge: int) -> Image.Image:
        current_long_edge = max(image.size)
        if current_long_edge == long_edge:
            return image
        scale = long_edge / current_long_edge
        size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        return image.resize(size, Image.Resampling.LANCZOS)

    def _encode(self, image: Image.Image) -> bytes:
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=94, optimize=True, progressive=True)
        return buffer.getvalue()

def data_url_from_bytes(image_bytes: bytes, content_type: str = "image/jpeg") -> str:
    return f"data:{content_type};base64,{base64.b64encode(image_bytes).decode('ascii')}"

def make_side_by_side(original: Image.Image, enhanced: Image.Image, panel_width: int = 480) -> Image.Image:
    def fit(image: Image.Image) -> Image.Image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        height = max(1, round(panel_width * image.height / image.width))
        return image.resize((panel_width, height), Image.Resampling.LANCZOS)

    left = fit(original)
    right = fit(enhanced)
    label_height = 42
    gap = 20
    canvas = Image.new("RGB", (panel_width * 2 + gap, max(left.height, right.height) + label_height), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((0, 12), "Original", fill=(20, 23, 20))
    draw.text((panel_width + gap, 12), "Enhanced", fill=(20, 23, 20))
    canvas.paste(left, (0, label_height))
    canvas.paste(right, (panel_width + gap, label_height))
    return canvas

print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
print("Pipeline ready.")

In [ ]:
# @title 4. Upload an inventory image
from google.colab import files
from IPython.display import display

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

input_name, input_bytes = next(iter(uploaded.items()))
original_image = Image.open(io.BytesIO(input_bytes))
print(f"Uploaded: {input_name} | {len(input_bytes) / 1024:.1f} KB | {original_image.width} x {original_image.height}")
display(original_image)

In [ ]:
# @title 5. Enhance and compare
engine = "studio_product" # @param ["studio_product", "studio_product_generative", "realesrgan", "pillow_fallback"]
preset = "product_standard" # @param ["product_standard", "product_detail", "product_soft"]
target_long_edge = 1800 # @param {type:"slider", min:800, max:2400, step:100}

if engine == "studio_product_generative" and not os.getenv("OPENAI_API_KEY"):
    from getpass import getpass
    openai_api_key = getpass("Enter OPENAI_API_KEY for studio_product_generative: ").strip()
    if not openai_api_key:
        raise ValueError("studio_product_generative needs OPENAI_API_KEY. Use studio_product for the non-generative local mode.")
    os.environ["OPENAI_API_KEY"] = openai_api_key

enhancer = ProductImageEnhancer(WEIGHTS_PATH, target_long_edge=target_long_edge)
result = enhancer.enhance(input_bytes, preset=preset, engine=engine)

enhanced_image = Image.open(io.BytesIO(result.image_bytes))
output_path = Path("/content/enhanced_product.jpg")
output_path.write_bytes(result.image_bytes)

print(f"Engine: {result.engine_label}")
print(f"Output: {result.width} x {result.height}")
print(f"Saved to: {output_path}")

comparison = make_side_by_side(original_image, enhanced_image)
display(comparison)

In [ ]:
# @title 6. Create an API-style response and download the output
import json
from google.colab import files

api_response = {
    "image": data_url_from_bytes(result.image_bytes),
    "width": result.width,
    "height": result.height,
    "engine": result.engine,
    "engine_label": result.engine_label,
    "preset": result.preset,
}

response_path = Path("/content/enhancement_response.json")
response_path.write_text(json.dumps(api_response, indent=2))

print("API-style response written to:", response_path)
print("Enhanced image written to:", output_path)
files.download(str(output_path))